new cleaning cell frm start

In [ ]:
import pandas as pd
import numpy as np

# ===============================================================
# 0. LOAD + BASIC CLEAN
# ===============================================================
df = pd.read_csv("survey_results_public.csv", na_values=["NA", "NaN", " "])
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


cols_to_drop = ["ResponseId",
    "LanguagesHaveEntry", "LanguagesWantEntry",
    "DatabaseHaveEntry", "DatabaseWantEntry",
    "PlatformHaveEntry", "PlatformWantEntry",
    "WebframeHaveEntry", "WebframeWantEntry",
    "DevEnvHaveEntry", "DevEnvWantEntry",
    "OfficeStackHaveEntry", "OfficeStackWantEntry",
    "CommPlatformHaveEntr", "CommPlatformWantEntr",
    "SOTagsHaveEntry", "SOTagsWantEntry",
    "AIModelsHaveEntry", "AIModelsWantEntry",
    "AIExplain",
    "AIAgentKnowWrite", "AIAgentOrchWrite",
    "AIAgentObsWrite", "AIAgentExtWrite",
    "AIOpen", "TechEndorse_13_TEXT", "TechOppose_15_TEXT",
    "JobSatPoints_15_TEXT", "SO_Actions_15_TEXT", "SOTagsWant Entry",

]

df.drop(columns=[c for c in cols_to_drop if c in df.columns],
        inplace=True)

print("🔥 Dropped user-typed and free-text columns.")

# ===============================================================
# 1. MULTI-SELECT columns → COUNT number of selections (SAFE)
# ===============================================================



EXCLUDE_MULTI = [ "AIToolCurrently partially AI",
    "AIToolDon't plan to use AI for this task",
    "AIToolPlan to partially use AI",
    "AIToolPlan to mostly use AI",
    "AIToolCurrently mostly AI",
    "Employment", "Age", "EdLevel", "LearnCodeChoose",
    "LearnCodeAI", "AIFrustration", "AIHuman", "AISelect"
]

multi_select_cols = [
    col for col in df.columns
    if col not in EXCLUDE_MULTI
    and df[col].astype(str).str.contains(";").any()
]


for col in multi_select_cols:
    df[col] = df[col].apply(lambda x: len(str(x).split(";")) if pd.notna(x) else 0)

print("✔ Multi-select encoded, AIHuman left untouched")

# ===============================================================
# Encode AISelect as binary (0 = no, 1 = yes)
# ===============================================================

ai_select_binary_map = {
    "Yes, I use AI tools daily": 1,
    "Yes, I use AI tools weekly": 1,
    "Yes, I use AI tools monthly or infrequently": 1,
    "No, but I plan to soon": 0,
    "No, and I don't plan to": 0
}

def clean_text(x):
    if not isinstance(x, str):
        return x
    return x.replace("’", "'").strip()

if "AISelect" in df.columns:
    df["AISelect"] = df["AISelect"].apply(clean_text).map(ai_select_binary_map).astype("Int64")
    print("✔ AISelect encoded as binary (0/1)")
else:
    print("⚠ AISelect column not found")

# ===============================================================
# Encode AIHuman correctly (multi-select → count → 0/1/2)
# ===============================================================

AIHUMAN_OPTIONS = {
    "when i'm stuck and can't explain the problem",
    "when i want to fully understand something",
    "when i want to learn best practices",
    "when i don’t trust ai’s answers",
    "when i need help fixing complex or unfamiliar code",
    "when i need quick help troubleshooting",
    "when i want to compare different solutions",
    "when i have ethical or security concerns about code",
    "i don’t think i’ll need help from people anymore"
}

def normalize_text(t):
    if not isinstance(t, str):
        return ""
    return t.replace("’", "'").lower().strip()

def count_aihuman(entry):
    if pd.isna(entry):
        return 0
    parts = [normalize_text(p) for p in entry.split(";")]
    valid = [p for p in parts if p in AIHUMAN_OPTIONS]
    return len(valid)

def collapse_aihuman(n):
    if n == 0: return 0
    elif n <= 3: return 1
    else: return 2

if "AIHuman" in df.columns:
    df["AIHuman_Count"] = df["AIHuman"].apply(count_aihuman)
    df["AIHuman_Class"] = df["AIHuman_Count"].apply(collapse_aihuman).astype("Int64")
    print(df["AIHuman_Class"].value_counts(dropna=False))
else:
    print("❌ AIHuman column not found")

# ---------- MainBranch ----------
if "MainBranch" in df.columns:
    def map_mainbranch(s):
        if pd.isna(s): return pd.NA
        s2 = s.lower()
        if "hobby" in s2: return 0
        if "learning to code" in s2: return 1
        if "not primarily a developer" in s2: return 2
        if "developer by profession" in s2: return 3
        return pd.NA
    df["MainBranchOrd"] = df["MainBranch"].apply(map_mainbranch).astype("Int64")
    df.drop(columns=["MainBranch"], inplace=True)

# ===============================================================
# SOPartFreq → proper ordinal encoding (0 → 7)
# ===============================================================

sopart_map = {
    "I have never participated in Q&A on Stack Overflow": 0,
    "Infrequently, less than once per year": 1,
    "Less than once every 2 - 3 months": 2,
    "Less than once per month or monthly": 3,
    "A few times per month or weekly": 4,
    "A few times per week": 5,
    "Daily or almost daily": 6,
    "Multiple times per day": 7
}

if "SOPartFreq" in df.columns:
    df["SOPartFreq"] = df["SOPartFreq"].map(sopart_map)
    print("✔ SOPartFreq encoded 0–7")
else:
    print("⚠ SOPartFreq not found")
# ===============================================================
# SOVisitFreq → proper ordinal encoding (1 → 7)
# ===============================================================

sovisit_map = {
    "Infrequently, less than once per year": 1,
    "Less than once every 2 - 3 months": 2,
    "Less than once per month or monthly": 3,
    "A few times per month or weekly": 4,
    "A few times per week": 5,
    "Daily or almost daily": 6,
    "Multiple times per day": 7
}

if "SOVisitFreq" in df.columns:
    df["SOVisitFreq"] = df["SOVisitFreq"].map(sovisit_map)
    print("✔ SOVisitFreq encoded 1–7")
else:
    print("⚠ SOVisitFreq not found in dataframe")


# ===============================================================
# Correct AIAgents encoding (0–5)
# ===============================================================

aiagents_map = {
    "No, and I don't plan to": 0,
    "No, but I plan to": 1,
    "No, I use AI exclusively in copilot/autocomplete mode": 2,

    "Yes, I use AI agents at work monthly or infrequently": 3,
    "Yes, I use AI agents at work weekly": 4,
    "Yes, I use AI agents at work daily": 5,
}

if "AIAgents" in df.columns:
    df["AIAgents"] = df["AIAgents"].map(aiagents_map)
    print("✔ AIAgents encoded 0–5")
else:
    print("⚠ AIAgents not found")
# ===============================================================
# Correct AIAgentChange encoding (0–3)
# ===============================================================

agentchange_map = {
    "Yes, to a great extent": 3,
    "Yes, somewhat": 2,
    "Not at all or minimally": 1,

    "No, but my development work has changed somewhat due to non-AI factors": 0,
    "No, but my development work has significantly changed due to non-AI factors": 0,
}

if "AIAgentChange" in df.columns:
    df["AIAgentChange"] = df["AIAgentChange"].map(agentchange_map)
    print("✔ AIAgentChange encoded 0–3")
else:
    print("⚠ AIAgentChange not found")




# ===============================================================
# SOFriction → correct ordinal encoding
# ===============================================================

sofriction_map = {
    "Rarely, almost never": 1,
    "Less than half of the time": 2,
    "About half of the time": 3,
    "More than half the time": 4,
    "I don't use AI or AI-enabled tools": 0
}

if "SOFriction" in df.columns:
    df["SOFriction"] = df["SOFriction"].map(sofriction_map)
    print("✔ SOFriction encoded correctly (0–4)")
else:
    print("⚠ SOFriction not found")


# ---------- RemoteWork ----------
if "RemoteWork" in df.columns:
    def map_remote(s):
        if pd.isna(s): return pd.NA
        s2 = s.lower()
        if "remote" in s2: return 2
        if "hybrid" in s2: return 1
        if "in-person" in s2 or "on-site" in s2: return 0
        return pd.NA
    df["RemoteWorkOrd"] = df["RemoteWork"].apply(map_remote).astype("Int64")
    df.drop(columns=["RemoteWork"], inplace=True)



# ===============================================================
# 3. FREQUENCY SCALE (SOFriction, etc.)
# ===============================================================

frequency_map = {
    "Rarely, almost never": 1,
    "About half of the time": 3,
    "A few times per month or weekly": 3,
    "A few times per week": 4,
    "Daily or almost daily": 5
}

freq_cols = [
    col for col in df.columns
    if df[col].astype(str).str.contains("times|rarely|daily|half", case=False).any()
]

for col in freq_cols:
    df[col] = df[col].map(frequency_map)


# ===============================================================
# 4. TRUST / SENTIMENT / FAVORABILITY
# ===============================================================

trust_map = {
    "Highly distrust": 1,
    "Somewhat distrust": 2,
    "Neither trust nor distrust": 3,
    "Somewhat trust": 4,
    "Highly trust": 5,
}

sent_map = {
    "Very unfavorable": 1,
    "Unfavorable": 2,
    "Indifferent": 3,
    "Favorable": 4,
    "Very favorable": 5,
}

if "AIAcc" in df.columns:
    df["AIAcc"] = df["AIAcc"].map(trust_map)

if "AISent" in df.columns:
    df["AISent"] = df["AISent"].map(sent_map)


# ===============================================================
# 5. COMPLEXITY SCALE
# ===============================================================

complexity_map = {
    "Bad at handling complex tasks": 1,
    "Neither good or bad at handling complex tasks": 3,
    "Good, but not great at handling complex tasks": 4,
}

if "AIComplex" in df.columns:
    df["AIComplex"] = df["AIComplex"].map(complexity_map)



# ===============================================================
# 7. AI AGENT USES / CHALLENGES (multi-select) → count
# ===============================================================

agent_multi_cols = [
    col for col in df.columns
    if "AIAgent" in col and df[col].astype(str).str.contains(";").any()
]

for col in agent_multi_cols:
    df[col] = df[col].apply(lambda x: len(str(x).split(";")) if pd.notna(x) else 0)


# ===============================================================
# 8. AI MODEL CHOICE (ChatGPT, Gemini, Copilot…) → count
# ===============================================================

model_cols = ["AIHuman", "AIOpen"]

for col in model_cols:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: len(str(x).split(";")) if pd.notna(x) else 0)

# ===============================================================
# Employment → label encoded (0–6)
# ===============================================================

def encode_employment(x):
    if pd.isna(x):
        return pd.NA

    s = x.strip().lower()

    if s == "not employed":
        return 0
    if s == "student":
        return 1
    if "independent contractor" in s or "self-employed" in s or "freelancer" in s:
        return 2
    if s == "employed":
        return 3
    if s == "retired":
        return 4
    if "prefer not to say" in s:
        return 6
    if "other" in s:   # catch open-text "Other (please specify):"
        return 5

    # fallback for any unexpected responses
    return 5

if "Employment" in df.columns:
    df["Employment"] = df["Employment"].apply(encode_employment).astype("Int64")
    print("✔ Employment encoded 0–6")






/tmp/ipython-input-4239646163.py:7: DtypeWarning: Columns (56,74,92,97,98,105,109,110,132,162,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("survey_results_public.csv", na_values=["NA", "NaN", " "])
/tmp/ipython-input-4239646163.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


🔥 Dropped user-typed and free-text columns.
✔ Multi-select encoded, AIHuman left untouched
✔ AISelect encoded as binary (0/1)
AIHuman_Class
0    21830
2    14020
1    13341
Name: count, dtype: Int64
✔ SOPartFreq encoded 0–7
✔ SOVisitFreq encoded 1–7
✔ AIAgents encoded 0–5
✔ AIAgentChange encoded 0–3
✔ SOFriction encoded correctly (0–4)
✔ Employment encoded 0–6


In [ ]:
import re
import numpy as np

# ===============================================================
# Clean free-typed year fields (YearsCode, YearsCodePro, WorkExp)
# ===============================================================

def coerce_years_freeform(v):
    """
    Safely parses free-text experience/year fields.
    Extracts numbers, handles vague terms, and caps > 50 → 51.
    """

    if pd.isna(v):
        return np.nan

    s = str(v).strip().lower()

    # 1) Extract any number inside the text ("10+ years", "60ish", "around 5")
    nums = re.findall(r"\d+\.?\d*", s)

    if len(nums) > 0:
        try:
            n = float(nums[0])
        except:
            return np.nan

        # Cap unrealistic values
        if n > 50:
            return 51

        # Treat small values (<1) as 0.5 (user means <1 year)
        if n < 1:
            return 0.5

        return n

    # 2) Handle written descriptions with no numbers
    if "less" in s:
        return 0.5

    if "few" in s or "some" in s:
        return 1

    if "more" in s:
        return 51

    if "decade" in s:
        return 10

    if "none" in s or "never" in s:
        return 0

    # Completely unparseable → NaN
    return np.nan


# ===============================================================
# Apply to all relevant columns
# ===============================================================

year_cols = ["YearsCode", "YearsCodePro", "WorkExp"]

for col in year_cols:
    if col in df.columns:
        df[col] = df[col].apply(coerce_years_freeform)
        print(f"✔ Cleaned {col} using free-text safe parser")
    else:
        print(f"⚠ {col} not found in dataframe")


✔ Cleaned YearsCode using free-text safe parser
⚠ YearsCodePro not found in dataframe
✔ Cleaned WorkExp using free-text safe parser


In [ ]:
# ===============================================================
#  ENCODING REMAINING COLUMNS (AIThreat, NewRole, Industry, etc.)
# ===============================================================

import numpy as np

# ----------------------------
# 1. AIThreat → ordinal
# ----------------------------

ai_threat_map = {
    "No": 0,
    "I'm not sure": 1,
    "Yes": 2
}

if "AIThreat" in df.columns:
    df["AIThreat"] = df["AIThreat"].map(ai_threat_map)


# ----------------------------
# 2. NewRole → ordinal ranking
# ----------------------------

newrole_map = {
    "I have neither consider or transitioned into a new career or industry": 0,
    "I have somewhat considered changing my career and/or the industry I work in": 1,
    "I have strongly considered changing my career and/or the industry I work in": 2,
    "I have transitioned into a new career and/or industry voluntarily": 3,
    "I have transitioned into a new career and/or industry involuntarily": 4
}

if "NewRole" in df.columns:
    df["NewRole"] = df["NewRole"].map(newrole_map)


# ----------------------------
# 3. Industry → label encoded
# ----------------------------

if "Industry" in df.columns:
    df["Industry"] = df["Industry"].astype("category").cat.codes


# ----------------------------
# 4. DevType → label encoded
# ----------------------------

if "DevType" in df.columns:
    df["DevType"] = df["DevType"].astype("category").cat.codes


# ----------------------------
# 5. OrgSize → ordinal (employees)
# ----------------------------

orgsize_map = {
    "Less than 20 employees": 1,
    "20 to 99 employees": 2,
    "100 to 499 employees": 3,
    "500 to 999 employees": 4,
    "1,000 to 4,999 employees": 5,
    "5,000 to 9,999 employees": 6,
    "10,000 or more employees": 7
}

if "OrgSize" in df.columns:
    df["OrgSize"] = df["OrgSize"].map(orgsize_map)


# ----------------------------
# 6. ICorPM → binary
# ----------------------------

icorpm_map = {
    "Individual contributor": 0,
    "People manager": 1
}

if "ICorPM" in df.columns:
    df["ICorPM"] = df["ICorPM"].map(icorpm_map)


# ----------------------------
# 7. RemoteWork → ordinal
# ----------------------------

remotework_map = {
    "In-person": 0,
    "Hybrid (some in-person, leans heavy to flexibility)": 1,
    "Hybrid (some remote, some in-person but remote-heavy)": 2,
    "Remote": 3
}

if "RemoteWork" in df.columns:
    df["RemoteWork"] = df["RemoteWork"].map(remotework_map)


# ----------------------------
# 8. PurchaseInfluence → ordinal
# ----------------------------

purchase_map = {
    "No": 0,
    "Yes, I influenced the purchase of a tool that more than five colleagues use but it is not a substantial addition to the tech stack": 1,
    "Yes, I endorsed a tool that was open-source and is currently used by more than just myself but no purchase was made": 2,
    "Yes, I influenced the purchase of a substantial addition to the tech stack": 3
}

if "PurchaseInfluence" in df.columns:
    df["PurchaseInfluence"] = df["PurchaseInfluence"].map(purchase_map)


# ----------------------------
# 9. TechEndorseIntro → label encode
# ----------------------------

if "TechEndorseIntro" in df.columns:
    df["TechEndorseIntro"] = df["TechEndorseIntro"].astype("category").cat.codes


print("🔥 Remaining columns encoded successfully!")



🔥 Remaining columns encoded successfully!


In [ ]:
# ===============================================================
#  FINAL REMAINING COLUMNS: Age, EdLevel, Employment, LearnCodeChoose, LearnCodeAI
# ===============================================================

import numpy as np

# ----------------------------
# 1. Age → ordinal
# ----------------------------
AGE_ORDER = [
    "Under 18 years old","18-24 years old","25-34 years old",
    "35-44 years old","45-54 years old","55-64 years old","65 years or older"
]
age_to_ord = {k:i for i,k in enumerate(AGE_ORDER)}
age_to_mid = {
    "Under 18 years old":16,"18-24 years old":21,"25-34 years old":30,
    "35-44 years old":40,"45-54 years old":50,"55-64 years old":60,"65 years or older":70
}
if "Age" in df.columns:
    df["AgeOrd"] = df["Age"].map(age_to_ord).astype("Int64")
    df["AgeYearsMid"] = df["Age"].map(age_to_mid).astype("Int64")
    df.drop(columns=["Age"], inplace=True)






# ===============================================================
# EdLevel → Ordinal (0–6)
# ===============================================================

def encode_edlevel(x):
    if pd.isna(x):
        return pd.NA

    s = x.strip().lower()

    if "primary" in s or "elementary" in s:
        return 0

    if "secondary" in s or "high school" in s or "gymnasium" in s:
        return 1

    if "some college" in s or "without earning a degree" in s:
        return 2

    if "associate" in s:
        return 3

    if "bachelor" in s or "b.a." in s or "b.s." in s or "b.eng" in s:
        return 4

    if "master" in s or "m.a." in s or "m.s." in s or "m.eng" in s or "mba" in s:
        return 5

    if "professional" in s or "phd" in s or "ph.d" in s or "md" in s or "jd" in s or "ed.d" in s:
        return 6

    return pd.NA

df["EdLevel"] = df["EdLevel"].apply(encode_edlevel).astype("Int64")
print(df["EdLevel"].value_counts(dropna=False))




# ----------------------------
# 4. LearnCodeChoose → ordinal
# ----------------------------

learnchoose_map = {
    "Yes, I am new to coding or currently a student": 1,
    "Yes, I am not new to coding but am learning new coding techniques or programming language": 2,
    "No, I am not new to coding and did not learn new coding techniques or programming languages": 3,
}

if "LearnCodeChoose" in df.columns:
    df["LearnCodeChoose"] = df["LearnCodeChoose"].map(learnchoose_map)


# ----------------------------
# 5. LearnCodeAI → ordinal
# ----------------------------

learnai_map = {
    "Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies": 1,
    "Yes, I learned how to use AI-enabled tools required for my job or to benefit my career": 2,
    "No, I learned something that was not related to AI or AI enablement for my personal curiosity and/or hobbies": 3,
    "No, I learned something that was not related to AI or AI enablement as required for my job or to benefit my career": 4
}

if "LearnCodeAI" in df.columns:
    df["LearnCodeAI"] = df["LearnCodeAI"].map(learnai_map)


print("🔥 Final columns encoded: Age, EdLevel, Employment, LearnCodeChoose, LearnCodeAI")


EdLevel
4       20278
5       12589
2        6182
1        3631
6        2624
<NA>     1743
3        1562
0         582
Name: count, dtype: Int64
🔥 Final columns encoded: Age, EdLevel, Employment, LearnCodeChoose, LearnCodeAI


In [ ]:
# ===============================================================
# FIND REMAINING CATEGORICAL (NON-NUMERIC) COLUMNS
# ===============================================================

remaining_cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("🟦 REMAINING NON-NUMERIC COLUMNS:")
for col in remaining_cat_cols:
    print(f"- {col}")


🟦 REMAINING NON-NUMERIC COLUMNS:
- Country
- Currency
- LanguageChoice
- DatabaseChoice
- PlatformChoice
- WebframeChoice
- DevEnvsChoice
- AIModelsChoice
- SOAccount
- SODuration
- SOComm
- AIToolCurrently partially AI
- AIToolDon't plan to use AI for this task
- AIToolPlan to partially use AI
- AIToolPlan to mostly use AI
- AIToolCurrently mostly AI
- AIFrustration


In [ ]:
# ===============================================================
# PRINT UNIQUE VALUES FOR EACH CATEGORICAL COLUMN
# ===============================================================

for col in remaining_cat_cols:
    print(f"\n🔹 COLUMN: {col}")
    print(df[col].dropna().unique()[:50])  # print first 50 unique values



🔹 COLUMN: Country
['Ukraine' 'Netherlands' 'India' 'Georgia' 'Australia' 'Greece' 'Germany'
 'Bangladesh' 'Brazil' 'United States of America' 'Lithuania'
 'United Kingdom of Great Britain and Northern Ireland' 'Ireland' 'Sweden'
 'Dominican Republic' 'Austria' 'Belgium' 'Czech Republic' 'Italy'
 'Hungary' 'Malaysia' 'Switzerland' 'Egypt' 'Sri Lanka' 'Poland' 'Spain'
 'Russian Federation' 'Serbia' 'Japan' 'France' 'Romania' 'Canada'
 'Uruguay' 'United Arab Emirates' 'Argentina' 'Norway' 'Slovakia'
 'Republic of Moldova' 'Peru' 'Portugal' 'Costa Rica' 'Croatia'
 'Iran, Islamic Republic of...' 'Philippines' 'China' 'Finland' 'Colombia'
 'Ethiopia' 'Israel' 'Bulgaria']

🔹 COLUMN: Currency
['EUR European Euro' 'UAH Ukrainian hryvnia' 'USD United States dollar'
 'INR Indian rupee' 'AUD\tAustralian dollar' 'BDT\tBangladeshi taka'
 'BRL Brazilian real' 'GBP Pound sterling' 'SEK\tSwedish krona'
 'CZK\tCzech koruna' 'PLN Polish zloty' 'HUF\tHungarian forint'
 'MYR\tMalaysian ringgit' 'CHF\tSwis

In [ ]:
# ===============================================================
# ENCODE REMAINING CATEGORICAL COLUMNS PROPERLY
# ===============================================================

import numpy as np

# ----------------------------
# Simple Yes/No → binary
# ----------------------------

binary_cols = [
    "LanguageChoice",
    "DatabaseChoice",
    "PlatformChoice",
    "WebframeChoice",
    "DevEnvsChoice",
    "AIModelsChoice"
]

for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].map({"Yes": 1, "No": 0})


# ----------------------------
# AIModelsHaveEntry → label encode (too many unique)
# ----------------------------

from sklearn.preprocessing import LabelEncoder

if "AIModelsHaveEntry" in df.columns:
    le = LabelEncoder()
    df["AIModelsHaveEntry"] = le.fit_transform(df["AIModelsHaveEntry"].astype(str))


# ----------------------------
# SOAccount → ordinal
# ----------------------------

so_account_map = {
    "No": 0,
    "Not sure/can't remember": 1,
    "Yes": 2
}

if "SOAccount" in df.columns:
    df["SOAccount"] = df["SOAccount"].map(so_account_map)


# ----------------------------
# SODuration → ordinal
# ----------------------------

soduration_map = {
    "Less than one year": 1,
    "Between 1 and 3 years": 2,
    "Between 3 and 5 years": 3,
    "Between 5 and 10 years": 4,
    "Between 10 and 15 years": 5,
    "More than 15 years, or since Stack Overflow started in 2008": 6,
    "I don't use Stack Overflow": 0
}

if "SODuration" in df.columns:
    df["SODuration"] = df["SODuration"].map(soduration_map)


# ----------------------------
# SOComm → ordinal sentiment
# ----------------------------

socomm_map = {
    "No, not at all": 0,
    "No, not really": 1,
    "Not sure": 2,
    "Neutral": 3,
    "Yes, somewhat": 4,
    "Yes, definitely": 5
}

if "SOComm" in df.columns:
    df["SOComm"] = df["SOComm"].map(socomm_map)


# ----------------------------
# OfficeStackWantEntry → category encode (label encode)
# ----------------------------

if "OfficeStackWantEntry" in df.columns:
    le2 = LabelEncoder()
    df["OfficeStackWantEntry"] = le2.fit_transform(df["OfficeStackWantEntry"].astype(str))


print("🔥 All remaining categorical columns encoded cleanly!")


🔥 All remaining categorical columns encoded cleanly!


In [ ]:
df.select_dtypes(include=["object"]).columns


Index(['Country', 'Currency', 'AIToolCurrently partially AI',
       'AIToolDon't plan to use AI for this task',
       'AIToolPlan to partially use AI', 'AIToolPlan to mostly use AI',
       'AIToolCurrently mostly AI', 'AIFrustration'],
      dtype='object')

In [ ]:
df.drop(columns=[
    "Country",
    "Currency",
    "CompTotal",
    "ConvertedCompYearly"
], inplace=True)


ENCODING TARGETS

FRUSTRATION

In [ ]:
# ===============================================================
# AIFrustration → Normalize → Count → 3-class target
# ===============================================================

def normalize_quotes(text):
    if not isinstance(text, str):
        return text
    # Fix smart quotes + mojibake + lowercase + strip
    text = (
        text.replace("â€™", "'")
            .replace("â€˜", "'")
            .replace("’", "'")
            .replace("‘", "'")
            .replace("â€œ", '"')
            .replace("â€", '"')
            .replace("“", '"')
            .replace("”", '"')
    )
    return text.strip().lower()

# Normalize entire column BEFORE encoding
df["AIFrustration"] = df["AIFrustration"].astype(str).apply(normalize_quotes)

# Valid frustration options (already normalized)
frustration_options = {
    "ai solutions that are almost right, but not quite",
    "i don't use ai tools regularly",
    "debugging ai-generated code is more time-consuming",
    "i've become less confident in my own problem-solving",
    "i haven't encountered any problems",
    "it's hard to understand how or why the code works",
}

def count_frustration_items(entry):
    if pd.isna(entry) or entry.strip() == "":
        return 0

    # Split, normalize each item again, lowercase, strip
    items = [
        normalize_quotes(t)
        for t in entry.split(";")
        if t.strip() != ""
    ]

    # Keep only true frustration items (remove "other")
    cleaned = [t for t in items if t in frustration_options]

    return len(cleaned)

# Step 1 — count 0–6 frustrations
df["AIFrustration_Count"] = df["AIFrustration"].apply(count_frustration_items)

# Step 2 — collapse to 0/1/2 classes
def collapse_frustration(c):
    if c == 0:
        return 0           # no frustration
    elif c <= 2:
        return 1           # mild / moderate
    else:
        return 2           # high frustration

df["AIFrustration_Class"] = df["AIFrustration_Count"].apply(collapse_frustration).astype("Int64")

# Step 3 — rename final target
if "AIFrustration_Class" in df.columns:
    df.rename(columns={"AIFrustration_Class": "Frustration"}, inplace=True)
    print("✔ Renamed AIFrustration_Class → Frustration")
else:
    print("⚠️ Column AIFrustration_Class not found — rename skipped.")


✔ Renamed AIFrustration_Class → Frustration


JOBSAT

In [ ]:
# ===============================================================
# JobSat → numeric → 3 classes (0 = low, 1 = medium, 2 = high)
# ===============================================================

if "JobSat" in df.columns:
    print("Re-binning JobSat into 3 classes (0/1/2)...")

    def bin_jobsat(v):
        if pd.isna(v):
            return np.nan
        try:
            v = float(v)
        except:
            return np.nan

        if v <= 4:
            return 0   # low satisfaction
        elif v <= 7:
            return 1   # medium satisfaction
        else:
            return 2   # high satisfaction

    df["JobSat"] = df["JobSat"].apply(bin_jobsat).astype("Int64")

    print(df["JobSat"].value_counts(dropna=False))
else:
    print("⚠️ 'JobSat' not found — skipping.")


Re-binning JobSat into 3 classes (0/1/2)...
JobSat
<NA>    22521
2       13494
1       10716
0        2460
Name: count, dtype: Int64


JOBSPERSPECTIVECLASS

In [ ]:
# ===============================================================
# AI Tool Usage → JobPerspective & JobPerspectiveClass (0/1/2)
# Matches naming used in previous dataset + robust normalization
# ===============================================================

# --- 1. Normalize all column names to fix mojibake ---
df.columns = (
    df.columns
      .str.replace("â€™", "'", regex=False)
      .str.replace("’", "'", regex=False)
)

# --- 2. AITOOL source columns ---
AITOOL = [
    "AIToolCurrently partially AI",
    "AIToolCurrently mostly AI",
    "AIToolPlan to partially use AI",
    "AIToolPlan to mostly use AI",
    "AIToolDon't plan to use AI for this task",
]

# Make sure only existing columns are used
AITOOL = [c for c in AITOOL if c in df.columns]

if not AITOOL:
    print("⚠️ No AITOOL columns found in dataset — skipping JobPerspective.")
else:
    print("Using AITOOL columns:", AITOOL)

    # --- 3. Collapse map: 5 → 3 (low / medium / high) ---
    LEVEL_COLLAPSE = {
        "AIToolDon't plan to use AI for this task": 0,   # low adoption
        "AIToolCurrently partially AI": 1,               # moderate
        "AIToolPlan to partially use AI": 1,
        "AIToolCurrently mostly AI": 2,                  # strong adoption
        "AIToolPlan to mostly use AI": 2
    }

    # --- Helper to split multi-select lists ---
    def split_list(x):
        if pd.isna(x):
            return []
        return [t.strip() for t in str(x).split(";") if t.strip()]

    # --- 4. Check missingness ---
    missing_mask = df[AITOOL].isna()
    print("Rows with ALL AITOOL columns missing:",
          missing_mask.all(axis=1).sum())

    # --- 5. Collect all tool names actually present ---
    observed_tools = set()
    for c in AITOOL:
        for lst in df[c].dropna().apply(split_list):
            observed_tools.update(lst)

    if not observed_tools:
        print("⚠️ No tool values observed — can't build JobPerspective.")
        df["JobPerspective"] = np.nan
        df["JobPerspectiveClass"] = np.nan
    else:
        techs = sorted(observed_tools)

        # matrix: each column = a tool, rows = respondents
        M = pd.DataFrame(np.nan, index=df.index, columns=techs)

        # --- 6. Fill the score matrix based on collapse map ---
        for col in AITOOL:
            score = LEVEL_COLLAPSE[col]   # collapse (0/1/2)
            lists = df[col].fillna("").astype(str).str.split(";")

            for i, lst in lists.items():
                for raw in lst:
                    raw = raw.strip()
                    if not raw:
                        continue

                    prev = M.at[i, raw]
                    M.at[i, raw] = score if pd.isna(prev) else max(prev, score)

        # --- 7. Aggregate: median across tools ---
        df["JobPerspective"] = M.median(axis=1, skipna=True)

        # --- 8. Convert median → final class 0/1/2 ---
        df["JobPerspectiveClass"] = (
            df["JobPerspective"].round().astype("Int64")
        )

        print("✅ Created JobPerspective & JobPerspectiveClass (0 = low, 1 = mid, 2 = high)")
        print(df[["JobPerspective", "JobPerspectiveClass"]].head())


Using AITOOL columns: ['AIToolCurrently partially AI', 'AIToolCurrently mostly AI', 'AIToolPlan to partially use AI', 'AIToolPlan to mostly use AI', "AIToolDon't plan to use AI for this task"]
Rows with ALL AITOOL columns missing: 18125
✅ Created JobPerspective & JobPerspectiveClass (0 = low, 1 = mid, 2 = high)
   JobPerspective  JobPerspectiveClass
0             1.0                    1
1             1.0                    1
2             1.0                    1
3             1.0                    1
4             0.0                    0


In [ ]:
# ===============================================================
# FINAL CLEANUP: Replace all -1 values with NaN across dataset
# ===============================================================

df = df.replace(-1, np.nan)

print("✔ All -1 values replaced with NaN across entire dataset.")



# ===============================================================
# FINAL CLEANUP — DROP UNWANTED COLUMNS BEFORE SAVING
# ===============================================================

cols_to_drop_final = [
   "AIFrustration", "AgeYearsMid", "AIFrustration_Count",
   "AIThreat", "LanguageWantToWorkWith", "DatabaseWantToWorkWith", "PlatformWantToWorkWith",
   "WebframeWantToWorkWith","DevEnvsWantToWorkWith", "SOTagsWantToWorkWith", "OfficeStackAsyncWantToWorkWith",
   "CommPlatformWantToWorkWith", "AIModelsWantToWorkWith", "AISent", "AIAcc", "JobPerspective", "AIToolCurrently partially AI",
    "AIToolDon't plan to use AI for this task",
    "AIToolPlan to partially use AI",
    "AIToolPlan to mostly use AI",
    "AIToolCurrently mostly AI",
    # add any additional columns here
]

df.drop(
    columns=[c for c in cols_to_drop_final if c in df.columns],
    inplace=True
)

print("🧹 Final cleanup complete — dropped selected columns.")
print("Remaining columns:", len(df.columns))

df.to_csv("final_clean_dataset2025.csv", index=False)
print("✔ Saved as final_clean_dataset2025.csv")


✔ All -1 values replaced with NaN across entire dataset.
🧹 Final cleanup complete — dropped selected columns.
Remaining columns: 125
✔ Saved as final_clean_dataset2025.csv


In [ ]:
# Count -1 values per column
neg1_counts = (df == -1).sum()
print(neg1_counts[neg1_counts > 0])


Series([], dtype: Int64)


In [ ]:
'''import pandas as pd

df = pd.read_csv("survey_results_public.csv")

# Show unique values for EVERY categorical column
# (only for columns with dtype = object)
for col in df.select_dtypes(include="object").columns:
    print("\n" + "="*60)
    print(f"🔹 COLUMN: {col}")
    print("="*60)

    unique_vals = df[col].dropna().unique()

    # Print only first 50 unique values so output is manageable
    for v in unique_vals[:50]:
        print(" -", v)

    if len(unique_vals) > 50:
        print(f" ... ({len(unique_vals)} total unique values)")
'''

'import pandas as pd\n\ndf = pd.read_csv("survey_results_public.csv")\n\n# Show unique values for EVERY categorical column\n# (only for columns with dtype = object)\nfor col in df.select_dtypes(include="object").columns:\n    print("\n" + "="*60)\n    print(f"🔹 COLUMN: {col}")\n    print("="*60)\n\n    unique_vals = df[col].dropna().unique()\n\n    # Print only first 50 unique values so output is manageable\n    for v in unique_vals[:50]:\n        print(" -", v)\n\n    if len(unique_vals) > 50:\n        print(f" ... ({len(unique_vals)} total unique values)")\n'

MICE

In [ ]:
# ===============================================================
# FINAL STEP — MICE IMPUTATION + ROUNDING + TARGET CLIPPING
# ===============================================================

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import numpy as np

# ---------------------------------------------------------------
# 1. Copy cleaned dataset
# ---------------------------------------------------------------
df_clean = df.copy()

print("Before MICE: total missing =", df_clean.isna().sum().sum())

# ---------------------------------------------------------------
# 2. Numeric columns only
# ---------------------------------------------------------------
numeric_cols = df_clean.select_dtypes(include=["float64", "int64", "Int64"]).columns
X_numeric = df_clean[numeric_cols]

# ---------------------------------------------------------------
# 3. Create & run MICE imputer
# ---------------------------------------------------------------
mice = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=20,
    sample_posterior=False,
    random_state=42
)

X_imputed = mice.fit_transform(X_numeric)

# ---------------------------------------------------------------
# 4. Replace back into dataframe
# ---------------------------------------------------------------
df_clean[numeric_cols] = X_imputed

# ---------------------------------------------------------------
# 5. ROUND ALL numeric columns (fix decimals)
# ---------------------------------------------------------------
df_clean[numeric_cols] = df_clean[numeric_cols].round()

# ---------------------------------------------------------------
# 6. CLIP ONLY TARGET VARIABLES to 0–2
# ---------------------------------------------------------------
target_cols = ["Frustration", "JobPerspectiveClass", "JobSat"]

for col in target_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].clip(0, 2).astype("Int64")

print("After MICE + rounding + clipping, total missing =", df_clean.isna().sum().sum())

# ================================
# 🏷️ RENAME MAP — 2025 (SCHEMA ALIGNED)
# ================================

rename_map_2025 = {
    # ---- SAME AS PREVIOUS DATASETS ----
    "Employment": "employment",
    "MainBranchOrd": "developer_status",
    "OrgSize": "org_size",
    "WorkExp": "work_experience",
    "YearsCode": "coding_years",
    "AgeOrd": "age",
    "RemoteWorkOrd": "remote_work",

    "SOFriction": "so_friction",
    "SOVisitFreq": "so_frequency",
    "SOTagsHaveWorkedWith": "so_tags_used",
    "CommPlatformHaveWorkedWith": "comm_platforms_used",
    "LanguageHaveWorkedWith": "languages_used",

    "LearnCode": "learning_sources",

    "OpSysPersonal use": "personal_os_used",

    "AIComplex": "ai_complexity",

    # ---- JOB SAT POINTS (KEEP CONSISTENT STYLE) ----
    "JobSatPoints_1": "job_quality_priority",        # Control over quality
    "JobSatPoints_2": "job_autonomy_priority",       # Autonomy / trust
    "JobSatPoints_16": "job_manager_priority",       # Like your manager
    "JobSatPoints_3": "job_team_priority",           # Team collaboration
    "JobSatPoints_4": "job_mentors_priority",        # Expert mentors
    "JobSatPoints_5": "job_leadership_priority",     # Mentor / lead juniors
    "JobSatPoints_6": "job_expertise_priority",      # Specialized expertise
    "JobSatPoints_7": "job_impact_priority",         # Real-world problems
    "JobSatPoints_8": "job_stability_priority",      # Stability + growth
    "JobSatPoints_9": "job_innovation_priority",     # Challenging problems
    "JobSatPoints_10": "job_tech_priority",          # New tech/tools
    "JobSatPoints_11": "job_pay_priority",           # Pay & benefits
    "JobSatPoints_13": "job_peer_recognition_priority",
    "JobSatPoints_14": "job_leadership_recognition_priority",
    # ---- AI VARIABLES ----
    # ---------- Core AI ----------
    "AIModelsChoice":"llm_usage",          # uses LLMs or not
    "AILearnHow": "ai_learning_methods",    # how they learned AI
    "AIAgents": "ai_agent_usage",          # uses AI agents
    "AIAgentChange": "ai_workflow_change", # agents changed work style
    "AIAgent_Uses": "ai_agent_tasks",      # what tasks agents are used for
    "AIHuman": "human_help_scenarios",     # when humans preferred over AI

    # ---------- Agent Challenges ----------
    "AIAgentChallengesNeutral": "agent_challenges_neutral",
    "AIAgentChallengesStrongly agree": "agent_challenges_strong_agree",
    "AIAgentChallengesSomewhat agree": "agent_challenges_some_agree",
    "AIAgentChallengesStrongly disagree": "agent_challenges_strong_disagree",

    # ---------- Agent Impact ----------
    "AIAgentImpactStrongly agree": "agent_impact_strong_agree",
    "AIAgentImpactStrongly disagree": "agent_impact_strong_disagree",
    "AIAgentImpactSomewhat disagree": "agent_impact_some_disagree",

    # ---- TECH ATTITUDE ----
    "TechEndorse_8": "tech_reliability_priority",
    "TechEndorse_7": "tech_brand_priority",
    "TechEndorse_9": "tech_cost_priority",
    "TechEndorse_2": "tech_easy_api_priority",
    "TechEndorse_5": "tech_quality_priority",
    "TechEndorse_6": "tech_opensource_priority",
    "TechEndorse_1": "tech_ai_priority",
    "TechEndorse_3": "tech_robust_api_priority",
    "TechEndorse_4": "tech_customizable_priority",

    "TechOppose_1": "tech_no_ai_barrier",
    "TechOppose_2": "tech_usability_barrier",
    "TechOppose_3": "tech_api_barrier",
    "TechOppose_5": "tech_efficiency_barrier",
    "TechOppose_7": "tech_cost_barrier",
    "TechOppose_9": "tech_security_barrier",
    "TechOppose_11": "tech_better_options_barrier",
    "TechOppose_13": "tech_obsolete_barrier",
    "TechOppose_16": "tech_ethics_barrier",


    # ---- STACKOVERFLOW BEHAVIOR ----
    "SOComm": "so_community_member",
    "SO_Dev_Content": "so_alt_content_preference",
    "SOActions_3": "so_bookmark_activity",

    # ---- ROLE / EMPLOYMENT ----
    "NewRole": "career_change",
    "EmploymentAddl": "side_activities",

    # ---- LEARNING ----
    "LearnCodeAI": "ai_skill_learning",

    # ---- PERSONAL TOOLING ----
    "ToolCountPersonal": "personal_tool_count"
}

df_clean.rename(columns=rename_map_2025, inplace=True)

print(f"✅ 2025 variables renamed. Total columns now: {df.shape[1]}")
cols_to_drop_2025 = [

    # ---- Purchase / Decision ----
    "PurchaseInfluence",
    # ---- TEXT / WRITE-IN  ----
    "TechEndorseIntro",
    "TechEndorse_13",
    "TechEndorse_13_TEXT",
    "TechOppose_15",
    "TechOppose_15_TEXT",
    "JobSatPoints_15",
    "JobSatPoints_15_TEXT"]

existing_cols = [c for c in cols_to_drop_2025 if c in df_clean.columns]

df_clean.drop(columns=existing_cols, inplace=True, errors="ignore")
# ---------------------------------------------------------------
# 7. Save
# ---------------------------------------------------------------
df_clean.to_csv("devx_2025_modelling_Dataset.csv", index=False)
print("✔ Saved final cleaned + imputed dataset!")


Before MICE: total missing = 1390508


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


After MICE + rounding + clipping, total missing = 0
✅ 2025 variables renamed. Total columns now: 125
✔ Saved final cleaned + imputed dataset!
